# T5-small transfer learning on Kaggle

Fine-tune pretrained T5 on the same generative QA examples as the scratch models.

Enable **Internet** and a **GPU accelerator** in Kaggle settings. Edit `REPO_URL`
below after pushing this project to GitHub. The workflow uses GPU 0, even if
Kaggle exposes two GPUs. Long implementations live in `src/qa_assignment/`.

Artifacts go to `/kaggle/working/qa_assignment/`. Each variant and seed has its
own checkpoint directory. Kaggle working files are temporary: download the
archive or save the notebook outputs, then attach those outputs to resume a
later session. Checkpoints are written every 250 optimizer steps and each epoch.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = ""  # fill in your GitHub repository URL after pushing the project
REPO_REF = "main"  # preferably the same commit SHA in BOTH notebooks
SOURCE_SUBDIR = "."  # edit if this assignment lives below the repository root
WORKSPACE = Path("/kaggle/working")
CLONE_ROOT = WORKSPACE / "qa_assignment_source"
OUTPUT_ROOT = WORKSPACE / "qa_assignment"
RESTORE_FROM = None  # e.g. Path('/kaggle/input/prior-qa-output/qa_assignment')

if not REPO_URL.strip():
    raise ValueError("Set REPO_URL to the GitHub repository you pushed this project to.")
if not WORKSPACE.is_dir():
    raise RuntimeError("This notebook is intended for Kaggle. Run local tests with pytest.")
if not (CLONE_ROOT / ".git").is_dir():
    subprocess.run(["git", "clone", "--no-checkout", "--depth", "1", REPO_URL, str(CLONE_ROOT)], check=True)
subprocess.run(["git", "-C", str(CLONE_ROOT), "fetch", "--depth", "1", "origin", REPO_REF], check=True)
subprocess.run(["git", "-C", str(CLONE_ROOT), "checkout", "--detach", "FETCH_HEAD"], check=True)
REPO_ROOT = (CLONE_ROOT / SOURCE_SUBDIR).resolve()
assert (REPO_ROOT / "pyproject.toml").is_file(), "Check SOURCE_SUBDIR."
print("Source:", REPO_ROOT)
print("Artifacts:", OUTPUT_ROOT)
subprocess.run(["git", "-C", str(CLONE_ROOT), "rev-parse", "HEAD"], check=True)

## Install the Python helpers

Run setup before importing project modules. These requirements pin the Hugging
Face interface used here and preserve Kaggle's preinstalled CUDA-enabled PyTorch.
If you previously imported Transformers in this session, restart the session
after installing and rerun from the top. A successful import is followed by an
actual forward/backward and generation probe before training.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(REPO_ROOT / "requirements-kaggle.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                "-e", str(REPO_ROOT)], check=True)

import torch
from qa_assignment.utils import environment_info
from qa_assignment.runtime import restore_artifacts

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU accelerator in Kaggle settings.")
DEVICE = "cuda:0"
if RESTORE_FROM is not None:
    print(restore_artifacts(RESTORE_FROM, OUTPUT_ROOT))
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
environment_info(REPO_ROOT)

## Download the whole dataset and prepare identical splits

Both official SQuAD 1.1 files are downloaded in full. `MAX_TRAIN_EXAMPLES=None`
trains on all eligible training examples; set 10,000 for a shorter pilot project.
Examples are retained only when the full formatted question and context fit
the input cap. No answer-guided cropping occurs. Internal development articles
come exclusively from training, and official validation is reserved for final
reporting. Scores describe the retained subset.

Copy the exact `DataConfig` between notebooks. Compare `data_fingerprint` values
before comparing results. You can restore `data/` from previous output to reuse
tokenization, but both notebooks can independently reproduce the same split.

In [ ]:
from qa_assignment.config import DataConfig
from qa_assignment.data import prepare_data, QACollator

DATA_CONFIG = DataConfig(
    max_input_length=512,
    max_output_length=64,
    development_fraction=0.1,
    split_seed=2026,
    max_train_examples=None,
)
bundle = prepare_data(OUTPUT_ROOT / "data", DATA_CONFIG)
collator = QACollator(bundle.tokenizer, DATA_CONFIG)
print("Data fingerprint:", bundle.manifest["data_fingerprint"])
print("Prepared cache:", bundle.directory)
print("Train / development / validation:", len(bundle.train), len(bundle.development), len(bundle.validation))
bundle.manifest["statistics"]

## Fine-tuning configuration

T5 is a separate transfer-learning reference. It keeps the same tokenizer,
input/output limits, article partition, and example IDs, while using pretrained
weights and its own declared fine-tuning recipe. It does not require Mamba.

In [ ]:
from qa_assignment.config import ModelConfig, TrainConfig, BenchmarkConfig

MODEL_CONFIG = ModelConfig(variant="t5_small", t5_name="google-t5/t5-small")
TRAIN_CONFIG = TrainConfig(epochs=3, micro_batch_size=4, accumulation_steps=4,
                           eval_batch_size=8, learning_rate=1e-4, precision="fp32",
                           save_every_steps=250, development_limit=None, seed=42)
BENCHMARK_CONFIG = BenchmarkConfig(examples=200, repeats=3,
                                   throughput_batch_size=4, fixed_output_tokens=32)

## Compatibility and memory pilot

Download T5-small, run forward/backward and cached generation checks, and inspect
the measured memory/epoch-time estimate. If this does not fit, reduce microbatch
size and increase accumulation to retain the effective batch size.

In [ ]:
from qa_assignment.runtime import run_pilot
from qa_assignment.utils import write_json

pilot = run_pilot(bundle, MODEL_CONFIG, TRAIN_CONFIG, collator, DEVICE,
                  steps=50, repo_root=REPO_ROOT)
write_json(OUTPUT_ROOT / "results" / "t5_pilot.json", pilot)
pilot

## Fine-tune and evaluate

`checkpoints/t5_small/seed_42/` contains full resumable `last.pt`, the selected
`best.pt`, step snapshots, configuration, data manifest, and dependency versions.
After final evaluation, `hf_export/` contains the selected model and tokenizer in
Hugging Face format. Select checkpoints using internal development, then score
official validation and measure the same two inference workloads.

In [ ]:
from qa_assignment.workflow import run_experiment, archive_outputs

RUN_TRAINING = True
if RUN_TRAINING:
    try:
        summary = run_experiment(bundle, MODEL_CONFIG, TRAIN_CONFIG, OUTPUT_ROOT,
                                 device=DEVICE, resume="auto", benchmark_config=BENCHMARK_CONFIG,
                                 repo_root=REPO_ROOT)
        print(summary)
    finally:
        print("Available checkpoints/results archived at:", archive_outputs(OUTPUT_ROOT))
else:
    print("Training disabled; set RUN_TRAINING=True after inspecting the pilot.")

## Results and output archive

Results include normalized EM/F1, predictions, selected checkpoint, resource
measurements, and warmed timing repeats. Actual QA latency and forced 32-token
decoding are separate measurements. T5 and the controlled models are plotted
separately. Import prior `results/` and `checkpoints/` to see runs from the other
notebook. Matching data fingerprints are required for comparison.

In [ ]:
from qa_assignment.workflow import collect_results, plot_results, archive_outputs

table = collect_results(OUTPUT_ROOT)
display(table)
if not table.empty:
    figures = plot_results(table, OUTPUT_ROOT)
    import matplotlib.pyplot as plt
    plt.show()

In [ ]:
INCLUDE_DATA_IN_ARCHIVE = True  # enables reuse in the second notebook
archive_path = archive_outputs(OUTPUT_ROOT, include_data=INCLUDE_DATA_IN_ARCHIVE)
print("Download from Kaggle Output:", archive_path)
print("Checkpoint root:", OUTPUT_ROOT / "checkpoints")

## Resume in another Kaggle session

Save/download these outputs, attach the prior output as a Kaggle input, and set
`RESTORE_FROM` to the folder containing `checkpoints/`, `results/`, and optionally
`data/`. Restore before preparing new artifacts. The helper copies read-only
Kaggle inputs into writable working storage; existing folders are never overwritten.
`resume='auto'` loads each variant's `last.pt`. Keep the complete checkpoint folder,
including `best.pt`, and use identical model/data/training settings and source commit.

To start a different experiment, change `OUTPUT_ROOT`; changing settings inside an
existing run is rejected. `last.pt` resumes training; `best.pt` is selected using
internal-development F1. Load only checkpoints you created or otherwise trust.